# 01 — Data Exploration & EDA
**Deep Learning for Cryptocurrency Price Forecasting**
*Muluh Penn Junior Patrick — M.Tech. Thesis 2026*

---
This notebook explores the raw collected data for all five assets (BTC, ETH, SOL, SUI, XRP),
covering price distributions, volatility regimes, correlation structure, and stationarity.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DATA_DIR = Path('../data/raw/ohlcv')
ASSETS   = ['BTC', 'ETH', 'SOL', 'SUI', 'XRP']
COLORS   = ['#3266AD', '#1D9E75', '#D85A30', '#BA7517', '#7F77DD']

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.3})
print('Environment ready.')


## 1.1 Load raw OHLCV data

In [ ]:
dfs = {}
for asset in ASSETS:
    path = DATA_DIR / asset / f'{asset}_1d.parquet'
    if path.exists():
        dfs[asset] = pd.read_parquet(path, columns=['open','high','low','close','volume'])
        dfs[asset].index = pd.to_datetime(dfs[asset].index)
        print(f'  {asset}: {len(dfs[asset]):,} rows  '
              f'{dfs[asset].index[0].date()} → {dfs[asset].index[-1].date()}')
    else:
        print(f'  {asset}: not found at {path}')


## 1.2 Price history overview

In [ ]:
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(12, 10), sharex=True)
for ax, (asset, color) in zip(axes, zip(ASSETS, COLORS)):
    if asset in dfs:
        ax.plot(dfs[asset].index, dfs[asset]['close'] / 1000,
                color=color, linewidth=1, label=asset)
    ax.set_ylabel('Price (k USD)', fontsize=9)
    ax.legend(loc='upper left', fontsize=9)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.suptitle('Closing Price History — All Assets (Daily)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()


## 1.3 Log-return distributions

In [ ]:
fig, axes = plt.subplots(1, len(ASSETS), figsize=(14, 4))
for ax, (asset, color) in zip(axes, zip(ASSETS, COLORS)):
    if asset not in dfs:
        continue
    log_ret = np.log(dfs[asset]['close'] / dfs[asset]['close'].shift(1)).dropna()
    ax.hist(log_ret, bins=80, color=color, alpha=0.75, density=True)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(asset, fontsize=10)
    ax.set_xlabel('Log return')
    if ax == axes[0]:
        ax.set_ylabel('Density')
    stats = f'μ={log_ret.mean():.4f}\nσ={log_ret.std():.4f}\nkurt={log_ret.kurt():.2f}'
    ax.text(0.97, 0.97, stats, transform=ax.transAxes, va='top', ha='right', fontsize=7)
plt.suptitle('Log-Return Distributions', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## 1.4 Cross-asset correlation heatmap

In [ ]:
import seaborn as sns

returns = pd.DataFrame({
    asset: np.log(dfs[asset]['close'] / dfs[asset]['close'].shift(1)).dropna()
    for asset in ASSETS if asset in dfs
}).dropna()

corr = returns.corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Daily Log-Return Correlations', fontsize=11)
plt.tight_layout()
plt.show()


## 1.5 Volatility regime analysis (rolling 30d std)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for asset, color in zip(ASSETS, COLORS):
    if asset not in dfs:
        continue
    log_ret = np.log(dfs[asset]['close'] / dfs[asset]['close'].shift(1))
    vol = log_ret.rolling(30).std() * np.sqrt(252) * 100   # annualised %
    ax.plot(vol.index, vol, color=color, linewidth=1, alpha=0.85, label=asset)
ax.set_ylabel('Annualised volatility (%)')
ax.set_xlabel('Date')
ax.set_title('Rolling 30-Day Annualised Volatility')
ax.legend()
plt.tight_layout()
plt.show()


## 1.6 Stationarity tests (ADF)

In [ ]:
from statsmodels.tsa.stattools import adfuller

print(f'{"Asset":<8} {"ADF stat":>10} {"p-value":>10} {"Stationary?":>12}')
print('─' * 44)
for asset in ASSETS:
    if asset not in dfs:
        continue
    series = dfs[asset]['close'].dropna()
    stat, pval, _, _, _, _ = adfuller(series, autolag='AIC')
    result = 'Yes (p<0.05)' if pval < 0.05 else 'No'
    print(f'{asset:<8} {stat:>10.3f} {pval:>10.4f} {result:>12}')
print()
print('Note: ADF on price levels — expect non-stationary (unit root).')
print('Models trained on log returns, which are stationary.')


## 1.7 Data quality summary
Key observations from EDA:
- All five assets show right-skewed log-return distributions with excess kurtosis (fat tails)
- BTC and ETH are highly correlated (>0.85); SUI and SOL show moderate correlation with BTC
- Volatility is regime-dependent — 2021 bull run and 2022 bear market clearly visible
- Price series are non-stationary (ADF fails to reject unit root); log returns are stationary
- Training uses log returns as target; USD prices reconstructed for evaluation

**→ Next: Feature Engineering** (`02_feature_engineering_demo.ipynb`)
